In [ ]:
#Schema Involved
from IPython.display import Image
Image("../docs/model.png")

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.config import (
    CUTOFF_DAY, PREPROCESSOR_PATH,
    RANDOM_STATE, RAW_DATA_DIR, SPLIT_PATH, SPLIT_RATIOS,
)
from src.etl.load_raw import load_raw_tables
from src.etl.cohort_labels import build_cohort_and_labels
from src.etl.build_feature_table import build_feature_table, export_feature_dictionary
from src.etl.make_split import make_split
from src.preprocessing.pipeline import fit_and_save_pipeline, load_preprocessor

In [ ]:
tables = load_raw_tables(RAW_DATA_DIR)
for name, df_tbl in tables.items():
    print(f"{name:20s}: {df_tbl.shape}")
print("So gia tri thieu o imd_band (da doi '?'):", tables["studentInfo"]["imd_band"].isna().sum())

In [ ]:
cohort_df, excluded_df = build_cohort_and_labels(
    tables["studentInfo"], tables["studentRegistration"], cutoff_day=CUTOFF_DAY
)
print("Cohort shape  :", cohort_df.shape)
print("Excluded shape:", excluded_df.shape)
print(f"Ti le label_withdrawn: {cohort_df['label_withdrawn'].mean() * 100:.2f}%")

In [ ]:
full_df = build_feature_table(cutoff_day=CUTOFF_DAY, save_outputs=True)
print("Feature table shape:", full_df.shape)
print(f"Ti le label_withdrawn: {full_df['label_withdrawn'].mean() * 100:.2f}%")
print("imd_missing.sum():", int(full_df["imd_missing"].sum()))
print("Cac cot con NaN:\n", full_df.isna().sum()[full_df.isna().sum() > 0])

In [ ]:
split_df = make_split(full_df, ratios=SPLIT_RATIOS, seed=RANDOM_STATE, output_path=SPLIT_PATH)
print("Split shape:", split_df.shape)
print(split_df["split"].value_counts(normalize=True).round(4))

In [ ]:
dict_df = export_feature_dictionary(list(full_df.columns))
print("Feature dictionary shape:", dict_df.shape)
dict_df.head(10)

In [ ]:
train_df = full_df[full_df["split"] == "train"]
dev_df = full_df[full_df["split"] == "dev"]

fit_and_save_pipeline(train_df, output_path=PREPROCESSOR_PATH)

prep_linear = load_preprocessor("linear", model_path=PREPROCESSOR_PATH)
prep_tree = load_preprocessor("tree", model_path=PREPROCESSOR_PATH)

print("Linear dev shape:", prep_linear.transform(dev_df).shape)
print("PCA dev shape   :", prep_linear.transform_numeric(dev_df).shape)
print("Tree dev shape  :", prep_tree.transform(dev_df).shape)